# Problem 3. 두 손글씨 숫자의 합 맞히기

## 문제 개요
문서 자동화 팀은 손으로 쓴 숫자 두 개가 나란히 적힌 칸에서 **두 숫자의 합**을 읽어내려 합니다.
그런데 학습용 이미지에는 각 숫자가 무엇인지 기록되어 있지 않고 **합만** 적혀 있습니다.
이미지를 보고 두 숫자의 합(0~18)을 예측하세요.

## 파일 구조
```
Problem3/
├── problem3.ipynb      ← 이 노트북
├── submission.csv        ← 제출 파일 (직접 생성)
└── data/
    ├── train/               숫자 쌍 이미지 10,000장 (28×56, pair_xxxxx.png)
    ├── test/                숫자 쌍 이미지 4,000장
    ├── digits/              숫자 하나짜리 이미지 1,000장 (28×28)
    ├── train.csv            filename, sum
    ├── digits.csv           filename, digit
    └── sample_submission.csv
```

## 데이터
| 파일 | 설명 |
|---|---|
| `train/`, `train.csv` | 왼쪽·오른쪽에 숫자가 하나씩 있는 28×56 흑백 이미지와 **두 숫자의 합** |
| `test/` | 같은 형식의 이미지, 합 없음 |
| `digits/`, `digits.csv` | 숫자 하나짜리 28×28 이미지 1,000장과 **그 숫자** (0~9) |

### 데이터 특징
- 숫자 쌍 이미지는 항상 왼쪽 절반(열 0~27)과 오른쪽 절반(열 28~55)에 숫자가 하나씩 있습니다.
- 합 분포는 9를 중심으로 산 모양입니다 (0과 18은 드묾).
- 출처: MNIST 손글씨 숫자 (Yann LeCun 외, CC BY-SA 3.0)를 이어 붙여 대회용으로 재구성

## 규칙
- 제한 시간: 3문제 합계 **90분**
- 허용: pandas, scikit-learn, LightGBM, XGBoost, OpenCV, PyTorch 등 일반 라이브러리와 공개 사전학습 모델
- 금지: LLM API로 테스트 데이터를 직접 예측, 테스트 데이터 수작업 라벨링, 원본 공개 데이터셋에서 정답 조회
- 결과는 이 노트북과 같은 위치(`Problem3/`)에 **`submission.csv`** 로 저장

## 평가
**Accuracy (정확도)** — 합이 정확히 맞은 이미지의 비율.

## 초기 코드 (베이스라인)
아래 베이스라인은 28×56 이미지 전체로 합(19개 클래스)을 바로 분류합니다. 동작은 하지만 점수가 낮습니다.

In [ ]:
import numpy as np, pandas as pd, os
from PIL import Image
from sklearn.linear_model import LogisticRegression
DATA_DIR = "data"
train = pd.read_csv(f"{DATA_DIR}/train.csv")
test_files = sorted(os.listdir(f"{DATA_DIR}/test"))
def load(folder, names): return np.stack([np.asarray(Image.open(f"{DATA_DIR}/{folder}/{n}"), dtype=np.float32).ravel() / 255 for n in names])
# 28x56 이미지 전체로 '합'을 바로 분류 (19개 클래스)
X = load("train", train.filename[:3000])
clf = LogisticRegression(max_iter=300).fit(X, train["sum"][:3000])
pd.DataFrame({"filename": test_files, "sum": clf.predict(load("test", test_files))}).to_csv("submission.csv", index=False)

## 제출 형식
`submission.csv`

| filename | sum |
|---|---|
| test_00000.png | 7 |
| test_00001.png | 12 |

- 컬럼명 `filename`, `sum`, sum은 0~18 정수, `data/test/`의 모든 이미지 포함
- 포맷이 틀리면 0점 처리됩니다.

In [ ]:
import pandas as pd
sub = pd.read_csv("submission.csv")
print(sub.shape)
sub.head()